In [8]:
import pandas as pd
import glob
from pathlib import Path
import os
from datetime import datetime

# Configuration des chemins
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"/ "election"

GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dossier Silver : {SILVER_DIR}")
print(f"Dossier Gold : {GOLD_DIR}")

Dossier Silver : /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver
Dossier Gold : /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/election


In [9]:
# 1. Lister tous les fichiers silver des élections présidentielles
silver_files = list(SILVER_DIR.glob("*-pres-t*-commune-rhone-69-silver.csv"))

print(f"Nombre de fichiers trouvés : {len(silver_files)}")

# 2. Charger et concaténer les DataFrames
df_list = []
for file in silver_files:
    df = pd.read_csv(file, sep=";")
    df_list.append(df)

df_all = pd.concat(df_list, ignore_index=True)
print(f"Taille du jeu de données consolidé : {df_all.shape}")

Nombre de fichiers trouvés : 4
Taille du jeu de données consolidé : (7378, 30)


In [10]:
# Extraction des combinaisons uniques d'année et de tour
dim_election = df_all[['annee_election', 'tour']].drop_duplicates().reset_index(drop=True)

# Création d'une clé primaire (id_election)
dim_election['id_election'] = dim_election['annee_election'].astype(str) + "_T" + dim_election['tour'].astype(str)

# Réorganisation des colonnes
dim_election = dim_election[['id_election', 'annee_election', 'tour']]

print(f"Taille Dim_Election : {dim_election.shape}")
display(dim_election.head())

Taille Dim_Election : (4, 3)


,id_election,annee_election,tour
0,2017_T2,2017,2
1,2022_T1,2022,1
2,2022_T2,2022,2
3,2017_T1,2017,1


In [11]:
# Extraction des informations géographiques
cols_geo = ['code_departement', 'libelle_departement', 'code_commune', 'libelle_commune']
dim_commune = df_all[cols_geo].drop_duplicates().reset_index(drop=True)

# Création d'une clé primaire (id_commune) formatée (ex: 69-001)
dim_commune['id_commune'] = dim_commune['code_departement'].astype(str) + "-" + dim_commune['code_commune'].astype(str)

# Réorganisation
dim_commune = dim_commune[['id_commune', 'code_departement', 'libelle_departement', 'code_commune', 'libelle_commune']]

print(f"Taille Dim_Commune : {dim_commune.shape}")
display(dim_commune.head())

Taille Dim_Commune : (284, 5)


,id_commune,code_departement,libelle_departement,code_commune,libelle_commune
0,69-1,69,Rhône,1,Affoux
1,69-2,69,Rhône,2,Aigueperse
2,69-3,69,Rhône,3,Albigny-sur-Saône
3,69-4,69,Rhône,4,Alix
4,69-5,69,Rhône,5,Ambérieux


In [12]:
# Extraction des informations candidats
cols_candidat = ['nom', 'prenom', 'sexe', 'nuance']
dim_candidat = df_all[cols_candidat].drop_duplicates().reset_index(drop=True)

# Les candidats peuvent avoir des nuances différentes entre 2 élections,
# Selon le modèle choisi, on peut agréger par nom/prenom. Ici on garde la combinaison complète.
dim_candidat['id_candidat'] = dim_candidat.index + 1 # Clé primaire auto-incrémentée
dim_candidat['id_candidat'] = "CAND_" + dim_candidat['id_candidat'].astype(str).str.zfill(3)

# Réorganisation
dim_candidat = dim_candidat[['id_candidat', 'nom', 'prenom', 'sexe', 'nuance']]

print(f"Taille Dim_Candidat : {dim_candidat.shape}")
display(dim_candidat.head())

Taille Dim_Candidat : (16, 5)


,id_candidat,nom,prenom,sexe,nuance
0,CAND_001,MACRON,Emmanuel,M,REM
1,CAND_002,LE PEN,Marine,F,RN
2,CAND_003,ARTHAUD,Nathalie,F,EXG
3,CAND_004,PÉCRESSE,Valérie,F,LR
4,CAND_005,POUTOU,Philippe,M,EXG


In [13]:
# Jointures pour récupérer les clés primaires (Foreign Keys)
fact_df = df_all.copy()

# 1. Jointure avec Dim_Election
fact_df['id_election'] = fact_df['annee_election'].astype(str) + "_T" + fact_df['tour'].astype(str)

# 2. Jointure avec Dim_Commune
fact_df['id_commune'] = fact_df['code_departement'].astype(str) + "-" + fact_df['code_commune'].astype(str)

# 3. Jointure avec Dim_Candidat
fact_df = fact_df.merge(
    dim_candidat, 
    on=['nom', 'prenom', 'sexe', 'nuance'], 
    how='left'
)

# 4. Sélection des métriques et des clés (on supprime les colonnes textuelles qui sont maintenant dans les dimensions)
cols_metriques = [
    'inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes',
    'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_blancs_vot',
    'pct_nuls_ins', 'pct_nuls_vot', 'pct_exprimes_ins', 'pct_exprimes_vot',
    'voix', 'pct_voix_ins', 'pct_voix_exprimes'
]

fact_resultats = fact_df[['id_commune', 'id_election', 'id_candidat'] + cols_metriques].copy()

# Ajout de métadonnées
fact_resultats['date_integration_gold'] = datetime.now().isoformat()

print(f"Taille Fact_Resultats : {fact_resultats.shape}")
display(fact_resultats.head())

Taille Fact_Resultats : (7378, 21)


,id_commune,id_election,id_candidat,inscrits,abstentions,votants,blancs,nuls,exprimes,pct_abs_ins,...,pct_blancs_ins,pct_blancs_vot,pct_nuls_ins,pct_nuls_vot,pct_exprimes_ins,pct_exprimes_vot,voix,pct_voix_ins,pct_voix_exprimes,date_integration_gold
0,69-1,2017_T2,CAND_001,257,39,218,20,7,191,15.18,...,7.78,9.17,2.72,3.21,74.32,87.61,93,36.19,48.69,2026-03-24T19:55:58.226396
1,69-1,2017_T2,CAND_002,257,39,218,20,7,191,15.18,...,7.78,9.17,2.72,3.21,74.32,87.61,98,38.13,51.31,2026-03-24T19:55:58.226396
2,69-2,2017_T2,CAND_001,224,51,173,13,11,149,22.77,...,5.80,7.51,4.91,6.36,66.52,86.13,84,37.50,56.38,2026-03-24T19:55:58.226396
3,69-2,2017_T2,CAND_002,224,51,173,13,11,149,22.77,...,5.80,7.51,4.91,6.36,66.52,86.13,65,29.02,43.62,2026-03-24T19:55:58.226396
4,69-3,2017_T2,CAND_001,1665,402,1263,110,26,1127,24.14,...,6.61,8.71,1.56,2.06,67.69,89.23,775,46.55,68.77,2026-03-24T19:55:58.226396


In [14]:
# Sauvegarde des dimensions
dim_election.to_csv(GOLD_DIR / "dim_election.csv", index=False, sep=";")
dim_commune.to_csv(GOLD_DIR / "dim_commune.csv", index=False, sep=";")
dim_candidat.to_csv(GOLD_DIR / "dim_candidat.csv", index=False, sep=";")

# Sauvegarde de la table de faits
fact_resultats.to_csv(GOLD_DIR / "fact_resultats.csv", index=False, sep=";")

print("✅ Modèle en étoile généré et sauvegardé avec succès dans le dossier data/gold/")

✅ Modèle en étoile généré et sauvegardé avec succès dans le dossier data/gold/
